# featurewiz, explained 2/4: SULOV, step by step
**SULOV** ("Searching for Uncorrelated List Of Variables", created by Ram Seshadri in 2018)
is featurewiz's first stage. In one sentence: *for every pair of highly correlated features, keep
the one that tells you more about the target.* We'll rebuild it in three steps and check
each step against the real featurewiz 0.6.1.

In [ ]:
import pathlib, sys

for module_file in pathlib.Path("/kaggle/input").glob("**/featurewiz_explained.py"):
    sys.path.insert(0, str(module_file.parent))
    break

import featurewiz_explained as fwx
import featurewiz_explained_plots as plots

> **featurewiz, explained**: a four-part series that rebuilds featurewiz 0.6.1's feature selection step by step.
> 1 Intuition · **2 SULOV** · 3 Recursive XGBoost · 4 Full pipeline.
> Everything here runs with Internet **off**. Results are checked against Reference runs of the real featurewiz 0.6.1.

In [ ]:
data = fwx.make_canonical_dataset()
features, target = data.features, data.target

## Step 1: find correlated pairs
A pair counts when the absolute Pearson correlation is **≥ 0.8**. Two Quirks:
- featurewiz() uses 0.8, but the SULOV function's own default is 0.70. Which you get depends on how you call it.
- The pairs are **sorted by name**, which affects ties later.

In [ ]:
corr = data.frame[features].corr().abs()
pairs = sorted(
    (features[i], features[j])
    for i in range(len(features)) for j in range(i + 1, len(features))
    if corr.iloc[i, j] >= 0.8
)
pairs

Three pairs, one per linear copy. `x4`–`x4_squared` isn't a pair: its correlation is

In [ ]:
round(corr.loc["x4", "x4_squared"], 3)

Pearson correlation only sees *straight-line* relationships, so SULOV can't catch a nonlinear copy.

## Step 2: score each paired feature by mutual information with the target
Mutual information measures how much knowing a feature tells you about the target, including nonlinear
relationships. featurewiz computes it only for features that are in some pair, with a fixed seed.

In [ ]:
from sklearn.feature_selection import mutual_info_classif

paired = list(dict.fromkeys(f for pair in pairs for f in pair))
mi = dict(zip(paired, mutual_info_classif(
    data.frame[paired], data.frame[target], n_neighbors=5, discrete_features=False, random_state=42)))
{f: round(v, 4) for f, v in sorted(mi.items(), key=lambda kv: -kv[1])}

Look at `x3` and `x3_copy`: the **copy** scores slightly higher. The estimate is noisy, and the copy wins by chance.

## Step 3: walk down the ranking, knocking out partners
Take the highest-scoring feature, keep it, and remove its partners from the list. Repeat.

In [ ]:
walk = sorted(mi, key=mi.get, reverse=True)
kept_paired, analysed = [], set()
for feature in walk:  # featurewiz removes items from the list while walking it; so do we
    kept_paired.append(feature)
    for a, b in pairs:
        pair = tuple(sorted((a, b)))
        if feature in (a, b) and pair not in analysed:
            analysed.add(pair)
            partner = b if feature == a else a
            if partner in walk:
                walk.remove(partner)
                print(f"keep {feature:8} (MI {mi[feature]:.4f}) -> knock out {partner} (MI {mi[partner]:.4f})")

kept = [f for f in features if f not in paired] + kept_paired
kept

Features in no pair go first, in their original order, then the winners in ranking order.
**This order matters in Lesson 3.**

## Check: our inline code, the module, and the real featurewiz

In [ ]:
reference = fwx.load_reference("canonical")
module_result = fwx.sulov(data.frame, features, target)
assert kept == module_result.kept  # same code, same environment: must agree
print("inline == module:", kept)
print("featurewiz 0.6.1:", reference.sulov_kept, "| same:", kept == reference.sulov_kept)

In [ ]:
plots.sulov_graph(module_result, data.frame).show()

## On real data: Breast Cancer Wisconsin
30 measurements of cell nuclei. Radius, perimeter and area are nearly the same measurement three times over.
One more Quirk: featurewiz **strips every character except letters, digits and `_` from column names**
(spaces included), so our copy of this dataset uses `concave_points_mean` instead of `concave points_mean`.

In [ ]:
bc = fwx.load_breast_cancer()
plots.correlation_heatmap(bc, fwx.BREAST_CANCER_FEATURES).show()

In [ ]:
bc_result = fwx.sulov(bc, fwx.BREAST_CANCER_FEATURES, "diagnosis")
print(f"SULOV keeps {len(bc_result.kept)} of 30:", bc_result.kept)
print("same as featurewiz 0.6.1:", bc_result.kept == fwx.load_reference("breast_cancer").sulov_kept)
plots.sulov_graph(bc_result, bc).show()

## What to remember
- SULOV only sees **linear** redundancy (`x4_squared` got through).
- Between copies it keeps the higher mutual information, which can be **the copy** (`x3_copy`).
- Its output order (unpaired first, then by mutual information) becomes Recursive XGBoost's column order in Lesson 3.